# VAIC2026 — Fine-tune TANG TRUY XUAT (e5 bi-encoder)

Vong 2 cho thay tran da dich: reranker da tan dung het nhung gi tang 1 dua len,
nhung **e5 chi dat Recall@1 0.6970** va de lot han 1/33 cau ra khoi top-5 —
reranker khong the cuu vi no chi sap xep lai. Nen lan nay train chinh **e5**.

- Loss: InfoNCE tren [positive] + [hard negative BM25 cung van ban] + [in-batch]
- LoRA r=16 tren query/key/value, merge_and_unload de luu model phang
- Test: **dung 33 cau da dong bang** -> so truc tiep voi e5 goc

In [ ]:
# Kaggle's torch (2.10+cu128) dropped Pascal sm_60 but the assigned GPU is often a
# P100 (sm_60) -> "no kernel image". Install a torch that supports P100 AND T4.
# MUST run before any `import torch`. Also drop torchvision/torchaudio (mismatched
# -> "torchvision::nms does not exist") and torchao (peft rejects Kaggle's 0.10.0).
!pip uninstall -y -q torchvision torchaudio torchao
!pip install -q rank-bm25 peft
!pip install -q torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0), '| sm_', torch.cuda.get_device_capability(0))
    x = torch.randn(8, device='cuda'); print('CUDA op OK:', float((x + 1).sum()))

In [ ]:
import glob

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input'
    return hits[0]

TRAIN_PY = find('train_retriever.py')
EVAL_PY  = find('eval_reranker.py')
CHUNKS   = find('chunks.jsonl')
EVAL_33  = find('eval_dienbien_v2.jsonl')
TRAIN_A  = find('train_dienbien_v2.jsonl')
OUT_E5   = '/kaggle/working/e5-dienbien'
for k in ['TRAIN_PY','EVAL_PY','CHUNKS','EVAL_33','TRAIN_A']:
    print(k, '=', eval(k))
# dem dong that -- dung tin ten file
for k, p in [('TRAIN_A', TRAIN_A), ('EVAL_33', EVAL_33)]:
    print(k, 'rows =', sum(1 for l in open(p, encoding='utf-8') if l.strip()))

## Train e5 — contrastive LoRA

In [ ]:
!python {TRAIN_PY}     --train {TRAIN_A} --out {OUT_E5}     --base-model intfloat/multilingual-e5-large     --use-lora --lora-r 16 --lora-alpha 32     --epochs 10 --batch-size 4 --grad-accum 2 --lr 1e-4     --max-len 320 --n-neg 4 --num-workers 2

## Eval tren 33 cau dong bang

Hai dong dang so la **e5 retrieval only** (goc) va **e5 fine-tuned** — do la
tang 1, truoc khi co bat ky rerank nao. Dong reranker goc giu lai lam moc.

In [ ]:
!python {EVAL_PY} --chunks {CHUNKS} --eval {EVAL_33}     --e5-model intfloat/multilingual-e5-large     --e5-finetuned {OUT_E5}     --base-model BAAI/bge-reranker-v2-m3     --finetuned /kaggle/working/__none__
import shutil; shutil.copy('eval_results.json', '/kaggle/working/eval_e5_finetune.json')

In [ ]:
import json
r = json.load(open('/kaggle/working/eval_e5_finetune.json'))
base = r.get('e5 retrieval only', {}); ft = r.get('e5 fine-tuned (retrieval only)', {})
print(json.dumps(r, indent=2, ensure_ascii=False))
if base and ft:
    print('
=== TANG 1: e5 goc -> e5 fine-tune (33 cau) ===')
    for k in ['Recall@1','Recall@3','Recall@5','MRR','nDCG@5']:
        d = ft[k]-base[k]
        print(f'  {k:9s} {base[k]:.4f} -> {ft[k]:.4f}   ({d:+.4f} = {d*33:+.1f} cau)')